In [ ]:
%pip install scikit-image
%pip install lpips
%pip install pytorch-fid
%pip install matplotlib
%pip install scikit-learn
%pip install torchmetrics

%pip uninstall opencv-python -y
%pip install opencv-contrib-python
%pip install torch-fidelity


In [2]:
import os
import tqdm
import numpy as np
# visualization
import matplotlib.pyplot as plt
from PIL import Image

# pytorch
import torch
from torchvision import transforms

import pytorch_fid.fid_score as fid
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity

In [3]:
path = '/home/jarvis-wang/my_project/csc-2541/'  # TODO: change to your path
origin_img_path = os.path.join(path, 'mae/img')

mae_img_path = os.path.join(path, 'mae/pred')
mae_gan_img_path = os.path.join(path, 'mae_gan/pred')
mae_gan_bound_percep_img_path = os.path.join(path, 'mae_ganbound_percep/pred')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cuda


In [4]:
def load_and_preprocess(image_path):
    """Load and preprocess image to tensor"""
    transform_steps = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    img = Image.open(image_path)
    return transform_steps(img)

def calculate_metrics(orig_tensor, pred_tensor):
    """Calculate LPIPS for a pair of images"""
    # Convert tensors for SSIM and PSNR calculation
    orig_np = orig_tensor.cpu().numpy().transpose(1, 2, 0)
    pred_np = pred_tensor.cpu().numpy().transpose(1, 2, 0)
    
    # Calculate LPIPS
    loss_fn = LearnedPerceptualImagePatchSimilarity(net_type='vgg').to(DEVICE)
    with torch.no_grad():
        lpips_score = loss_fn(
            orig_tensor.unsqueeze(0).to(DEVICE),
            pred_tensor.unsqueeze(0).to(DEVICE)
        ).item()
    
    return {
        'lpips': lpips_score,
    }

### LPIPS

In [5]:
def evaluate_model(model_name, orig_path, pred_path):
    """Evaluate all metrics for one model"""
    # Get all image files
    image_files = [f for f in os.listdir(orig_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    metrics = {
        'lpips': [],
    }

    for img_file in tqdm.tqdm(image_files, desc=f'Processing {model_name}'):
        orig_tensor = load_and_preprocess(os.path.join(orig_path, img_file))
        pred_tensor = load_and_preprocess(os.path.join(pred_path, img_file))   
        results = calculate_metrics(orig_tensor, pred_tensor)

        for metric_name, value in results.items():
            metrics[metric_name].append(value)
        
    # mean
    mean_metrics = {
        metric: np.mean(scores) for metric, scores in metrics.items()
    }
    # standard deviation
    std_metrics = {
        metric: np.std(scores) for metric, scores in metrics.items()
    }

    return mean_metrics, std_metrics

In [6]:
def evaluate_all_models():
    """Evaluate all models and print results"""
    models = {
        'MAE': mae_img_path,
        'MAE-GAN': mae_gan_img_path,
        'MAE-GAN-Bound-Percep': mae_gan_bound_percep_img_path
    }
    
    results = {}
    print("Evaluating models...")
    for model_name, pred_path in models.items():
        mean_metrics, std_metrics = evaluate_model(model_name, origin_img_path, pred_path)
        results[model_name] = {
            'mean': mean_metrics,
            'std': std_metrics
        }
    
    # Print results
    print("\nResults for all models:")
    print("-" * 50)
    for model_name, metrics in results.items():
        print(f"\n{model_name}:")
        print(f"LPIPS: {metrics['mean']['lpips']:.4f} ± {metrics['std']['lpips']:.4f}")
    print("-" * 50)
    print("Evaluation complete.")

    return results


In [7]:
results = evaluate_all_models()

Evaluating models...


Processing MAE:   0%|          | 0/60 [00:00<?, ?it/s]

/home/jarvis-wang/my_project/mat-1510/mat-1510-venv/lib/python3.10/site-packages/torchmetrics/functional/image/lpips.py:323: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  se


Results for all models:
--------------------------------------------------

MAE:
LPIPS: 0.3235 ± 0.1031

MAE-GAN:
LPIPS: 0.3391 ± 0.0487

MAE-GAN-Bound-Percep:
LPIPS: 0.3156 ± 0.0610
--------------------------------------------------
Evaluation complete.


### FID Score


In [8]:
from pytorch_fid import fid_score

def calculate_fid(orig_path, pred_path):
    """Calculate FID score between original and predicted images"""
    fid_value = fid_score.calculate_fid_given_paths(
        [orig_path, pred_path],
        batch_size=50,
        device=DEVICE,
        dims=2048
    )
    return fid_value

# Evaluate FID for each model
models = {
    'MAE': mae_img_path,
    'MAE-GAN': mae_gan_img_path,
    'MAE-GAN-Bound-Percep': mae_gan_bound_percep_img_path
}

for model_name, pred_path in models.items():
    fid_value = calculate_fid(origin_img_path, pred_path)
    print(f"{model_name} FID Score: {fid_value:.4f} (lower is better)")

100%|██████████| 2/2 [00:00<00:00,  9.14it/s]


MAE FID Score: 156.7875 (lower is better)


100%|██████████| 2/2 [00:00<00:00,  8.62it/s]


MAE-GAN FID Score: 122.0994 (lower is better)


100%|██████████| 2/2 [00:00<00:00,  8.35it/s]


MAE-GAN-Bound-Percep FID Score: 112.6038 (lower is better)
